In [0]:
from pyspark.sql import functions as F

roads_path = "/Volumes/workspace/default/urbanpulse_bronze/roads"

road_edges = spark.read.json(
    roads_path + "/road_edges_parts/*.json"
)

roads_silver = (
    road_edges
    .filter(F.col("properties").isNotNull())
    .withColumn("segment_id", F.col("properties.osmid"))
    .withColumn("road_name", F.col("properties.name"))
    .withColumn("highway_type", F.col("properties.highway"))
    .withColumn("max_speed", F.col("properties.maxspeed"))
    .withColumn("oneway", F.col("properties.oneway"))
    .withColumn("geometry_type", F.col("geometry.type"))
    .withColumn("coordinates", F.col("geometry.coordinates"))
    .drop("type", "properties", "geometry")
    .dropDuplicates()
)

roads_silver = roads_silver.filter(
    F.col("coordinates").isNotNull()
)

roads_silver.printSchema()

In [0]:
roads_silver = (
    roads_silver
    .withColumn("road_name", F.trim(F.col("road_name")))
    .withColumn(
        "highway_type",
        F.coalesce(F.col("highway_type"), F.lit("unclassified"))
    )
    .withColumn(
        "max_speed",
        F.regexp_extract(
            F.col("max_speed").cast("string"),
            r"(\d+)",
            1
        ).cast("integer")
    )
    .withColumn("oneway", F.col("oneway").cast("string"))
    .filter(F.col("coordinates").isNotNull())
    .dropDuplicates()
)

In [0]:
roads_silver = (
    roads_silver
    .withColumn(
        "road_category",
        F.when(F.col("highway_type").isin("motorway", "trunk"), "Major Road")
         .when(F.col("highway_type").isin("primary", "secondary"), "Main Road")
         .when(F.col("highway_type").isin("tertiary", "unclassified"), "Local Road")
         .otherwise("Other Road")
    )
    .withColumn("processed_at", F.current_timestamp())
)

In [0]:
(
    roads_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_road_edges")
)

In [0]:
road_nodes = spark.read.json(
    roads_path + "/road_nodes.geojson"
)

nodes_silver = (
    road_nodes
    .filter(F.col("geometry").isNotNull())
    .select(
        F.col("id").alias("node_id"),
        F.col("geometry.type").alias("geometry_type"),
        F.col("geometry.coordinates").alias("coordinates"),
        F.col("properties").alias("node_attributes")
    )
    .dropDuplicates()
)

In [0]:
nodes_silver = (
    road_nodes
    .filter(F.col("geometry").isNotNull())
    .select(
        F.col("properties.osmid").alias("node_id"),
        F.col("geometry.type").alias("geometry_type"),
        F.col("geometry.coordinates").alias("coordinates"),
        F.col("properties").alias("node_attributes")
    )
    .dropDuplicates()
)

(
    nodes_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_road_nodes")
)